# Spark RDD Project: Log Analytics, Fraud Detection, and API Monitoring

## Problem Statement
In a large e-commerce company, multiple systems generate huge volumes of logs every day. These logs contain user activity, authentication events, payment transactions, and API performance details. The company wants to analyze these logs to detect suspicious activity, identify repeated login and payment failures, monitor slow APIs, and generate meaningful business and operational insights.

In this project, we will use Apache Spark RDD to:
- read multiple log files
- parse raw log records
- clean malformed data
- analyze user actions and errors
- detect fraud and suspicious patterns
- monitor API performance
- build final alert and summary reports

In [3]:
from pyspark import SparkContext

# Create SparkContext
sc = SparkContext("local[*]", "RDD_Log_Analytics_Project")

# Read all files as RDDs
app_logs = sc.textFile("app_logs.txt")
auth_logs = sc.textFile("auth_logs.txt")
payment_logs = sc.textFile("payments_logs.txt")
api_logs = sc.textFile("api_gateway_logs.txt")

users_ref = sc.textFile("users_ref.csv")
ip_blacklist = sc.textFile("ip_blacklist.txt")

print("App Logs Sample:")
for row in app_logs.take(3):
    print(row)

print("\nAuth Logs Sample:")
for row in auth_logs.take(3):
    print(row)

print("\nPayment Logs Sample:")
for row in payment_logs.take(3):
    print(row)

print("\nAPI Logs Sample:")
for row in api_logs.take(3):
    print(row)

print("\nRecord Counts:")
print("App Logs Count:", app_logs.count())
print("Auth Logs Count:", auth_logs.count())
print("Payment Logs Count:", payment_logs.count())
print("API Logs Count:", api_logs.count())
print("Users Ref Count:", users_ref.count())
print("IP Blacklist Count:", ip_blacklist.count())

print("\nPartitions:")
print("App Logs Partitions:", app_logs.getNumPartitions())
print("Auth Logs Partitions:", auth_logs.getNumPartitions())
print("Payment Logs Partitions:", payment_logs.getNumPartitions())
print("API Logs Partitions:", api_logs.getNumPartitions())


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 18:04:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/10 18:04:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


App Logs Sample:


2026-03-05T10:00:05Z level=INFO user=user123 action=LOGIN_SUCCESS ip=192.168.1.10 device=web endpoint=/login ms=120 status=200
2026-03-05T10:00:12Z level=INFO user=user123 action=VIEW_PRODUCT ip=192.168.1.10 device=web endpoint=/product/432 ms=180 status=200
2026-03-05T10:00:20Z level=INFO user=user222 action=LOGIN_SUCCESS ip=192.168.1.12 device=android endpoint=/login ms=140 status=200

Auth Logs Sample:
2026-03-05T10:00:05Z user=user123 event=LOGIN_SUCCESS ip=192.168.1.10 reason=ok
2026-03-05T10:00:20Z user=user222 event=LOGIN_SUCCESS ip=192.168.1.12 reason=ok
2026-03-05T10:01:40Z user=user333 event=LOGIN_SUCCESS ip=192.168.1.45 reason=ok

Payment Logs Sample:
2026-03-05T10:02:25Z txn=tx101 user=user456 amount=1250 method=UPI status=FAILED ip=192.168.1.12
2026-03-05T10:03:15Z txn=tx102 user=user456 amount=1250 method=UPI status=FAILED ip=192.168.1.12
2026-03-05T10:05:22Z txn=tx103 user=user333 amount=899 method=CARD status=SUCCESS ip=192.168.1.45

API Logs Sample:
2026-03-05T10:00:12

## Section 3: Parsing Raw Logs

The log files we loaded are still raw text.

Example raw record:

2026-03-05T10:00:05Z level=INFO user=user123 action=LOGIN_SUCCESS ip=192.168.1.10 device=web endpoint=/login ms=120 status=200

Working with raw text is difficult for analysis.

For example:
- we cannot easily count users
- we cannot filter errors
- we cannot detect suspicious activity

Therefore we must **parse the logs**.

Parsing means converting raw text logs into **structured data**.

Example structured format:

{
    ts: 2026-03-05T10:00:05Z,
    level: INFO,
    user: user123,
    action: LOGIN_SUCCESS,
    ip: 192.168.1.10
}

We will use the **map() transformation** to apply a parsing function to every record in the RDD.

In [5]:
def parse_kv_log(line):
    # create dictionary
    #2026-03-05T10:00:05Z level=INFO user=user123 action=LOGIN_SUCCESS ip=192.168.1.10 device=web endpoint=/login ms=120 status=200
    parts=line.split()
    print(parts)
    record={}
    record["ts"]=parts[0]
    for item in parts[1:]:
        if "=" in item:
            key,value=item.split("=",1)
            record[key]=value
    return record

app_parsed=app_logs.map(parse_kv_log)
app_parsed = app_logs.map(parse_kv_log)
auth_parsed = auth_logs.map(parse_kv_log)
payment_parsed = payment_logs.map(parse_kv_log)
api_parsed = api_logs.map(parse_kv_log)


# Inspect parsed records

print("Parsed App Logs:")
for row in app_parsed.take(3):
    print(row)

Parsed App Logs:
{'ts': '2026-03-05T10:00:05Z', 'level': 'INFO', 'user': 'user123', 'action': 'LOGIN_SUCCESS', 'ip': '192.168.1.10', 'device': 'web', 'endpoint': '/login', 'ms': '120', 'status': '200'}
{'ts': '2026-03-05T10:00:12Z', 'level': 'INFO', 'user': 'user123', 'action': 'VIEW_PRODUCT', 'ip': '192.168.1.10', 'device': 'web', 'endpoint': '/product/432', 'ms': '180', 'status': '200'}
{'ts': '2026-03-05T10:00:20Z', 'level': 'INFO', 'user': 'user222', 'action': 'LOGIN_SUCCESS', 'ip': '192.168.1.12', 'device': 'android', 'endpoint': '/login', 'ms': '140', 'status': '200'}


['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']
['2026-03-05T10:00:12Z', 'level=INFO', 'user=user123', 'action=VIEW_PRODUCT', 'ip=192.168.1.10', 'device=web', 'endpoint=/product/432', 'ms=180', 'status=200']
['2026-03-05T10:00:20Z', 'level=INFO', 'user=user222', 'action=LOGIN_SUCCESS', 'ip=192.168.1.12', 'device=android', 'endpoint=/login', 'ms=140', 'status=200']


## Section 4: Data Cleaning and Record Validation

In real-world datasets, not every record is valid.

Some records may be:
- incomplete
- malformed
- missing fields
- incorrectly formatted

If we directly process such records, our analytics may produce incorrect results.

Therefore, a common data engineering practice is to separate data into two categories:

1. Valid Records → records that contain all required fields and can be safely analyzed
2. Invalid Records → malformed records that should be isolated for investigation

This process is called **data validation or data cleaning**.

In this section we will:
- validate parsed logs
- separate good records from bad records
- count and inspect invalid records

We will use the **filter() transformation** to achieve this.

In [29]:
record_1={'level': 'INFO', 'user': 'user123', 'action': 'LOGIN_SUCCESS', 'ip': '192.168.1.10', 'device': 'web', 'endpoint': '/login', 'ms': '120', 'status': '200'}

In [30]:
required_fields = ["ts","level","user","action","ip","endpoint","ms","status"]
for field in required_fields:
    if field not in record_1:
        print("False")

False


In [ ]:
# -----------------------------
# Validation functions
# -----------------------------

def is_valid_app(record):
    
    required_fields = ["ts","level","user","action","ip","endpoint","ms","status"]
    
    for field in required_fields:
        if field not in record:
            return False
    
    return True


def is_valid_auth(record):
    
    required_fields = ["ts","user","event","ip"]
    
    for field in required_fields:
        if field not in record:
            return False
    
    return True


def is_valid_payment(record):
    
    required_fields = ["ts","txn","user","amount","method","status","ip"]
    
    for field in required_fields:
        if field not in record:
            return False
    
    return True


def is_valid_api(record):
    
    required_fields = ["ts","endpoint","ip","ms","status"]
    
    for field in required_fields:
        if field not in record:
            return False
    
    return True


# -----------------------------
# Separate valid and invalid records
# -----------------------------

good_app = app_parsed.filter(is_valid_app) 
bad_app = app_parsed.filter(lambda x: not is_valid_app(x))

good_auth = auth_parsed.filter(is_valid_auth)
bad_auth = auth_parsed.filter(lambda x: not is_valid_auth(x))

good_payment = payment_parsed.filter(is_valid_payment)
bad_payment = payment_parsed.filter(lambda x: not is_valid_payment(x))

good_api = api_parsed.filter(is_valid_api)
bad_api = api_parsed.filter(lambda x: not is_valid_api(x))


# -----------------------------
# Record counts
# -----------------------------

print("Good App Records:", good_app.count())
print("Bad App Records:", bad_app.count())

print("Good Auth Records:", good_auth.count())
print("Bad Auth Records:", bad_auth.count())

print("Good Payment Records:", good_payment.count())
print("Bad Payment Records:", bad_payment.count())

print("Good API Records:", good_api.count())
print("Bad API Records:", bad_api.count())


# -----------------------------
# Inspect bad records
# -----------------------------

print("\nBad App Records Example:")
for row in bad_app.take(3):
    print(row)

print("\nBad Auth Records Example:")
for row in bad_auth.take(3):
    print(row)

['2026-03-05T10:06:00Z', 'level=ERROR', 'user=user222', 'action=DATABASE_TIMEOUT', 'ip=192.168.1.12', 'device=android', 'endpoint=/checkout', 'ms=2000', 'status=500']
['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']['2026-03-05T10:06:25Z', 'level=INFO', 'user=user123', 'action=LOGOUT_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/logout', 'ms=90', 'status=200']

['2026-03-05T10:07:01Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=130', 'status=401']
['2026-03-05T10:07:10Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=125', 'status=401']
['2026-03-05T10:00:12Z', 'level=INFO', 'user=user123', 'action=VIEW_PRODUCT', 'ip=192.168.1.10', 'device=web', 'endpoint=/product/432', 'ms=180', 'status=200']
['2026-03-05T10:07:20Z', 'level=ERROR', 'user=user88

Good App Records: 42


['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']
['2026-03-05T10:00:12Z', 'level=INFO', 'user=user123', 'action=VIEW_PRODUCT', 'ip=192.168.1.10', 'device=web', 'endpoint=/product/432', 'ms=180', 'status=200']
['2026-03-05T10:00:20Z', 'level=INFO', 'user=user222', 'action=LOGIN_SUCCESS', 'ip=192.168.1.12', 'device=android', 'endpoint=/login', 'ms=140', 'status=200']
['2026-03-05T10:00:31Z', 'level=INFO', 'user=user222', 'action=SEARCH', 'ip=192.168.1.12', 'device=android', 'endpoint=/search', 'ms=410', 'status=200']
['2026-03-05T10:01:02Z', 'level=WARN', 'user=user999', 'action=API_CALL', 'ip=192.168.1.99', 'device=web', 'endpoint=/search', 'ms=3100', 'status=200']
['2026-03-05T10:01:13Z', 'level=INFO', 'user=user123', 'action=ADD_TO_CART', 'ip=192.168.1.10', 'device=web', 'endpoint=/cart/add', 'ms=220', 'status=200']
['2026-03-05T10:01:40Z', 'level=INFO', 'user=user333', 'action=LO

Bad App Records: 1


['2026-03-05T10:07:40Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:50Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:08:05Z', 'user=user111', 'event=LOGIN_SUCCESS', 'ip=10.10.10.10', 'reason=ok']
['2026-03-05T10:08:06Z', 'user=user111', 'event=LOGIN_FAILED', 'ip=10.10.10.10', 'reason=token_replay']
['2026-03-05T10:08:07Z', 'user=user111', 'event=LOGIN_FAILED', 'ip=10.10.10.10', 'reason=token_replay']
['BROKEN_AUTH_LINE', 'user=user999', 'ip=1.2.3.4']
['2026-03-05T10:09:55Z', 'user=user888', 'event=LOGIN_SUCCESS', 'ip=192.168.1.55', 'reason=otp_success']
['2026-03-05T10:00:05Z', 'user=user123', 'event=LOGIN_SUCCESS', 'ip=192.168.1.10', 'reason=ok']
['2026-03-05T10:00:20Z', 'user=user222', 'event=LOGIN_SUCCESS', 'ip=192.168.1.12', 'reason=ok']
['2026-03-05T10:01:40Z', 'user=user333', 'event=LOGIN_SUCCESS', 'ip=192.168.1.45', 'reason=ok']
['2026-03-05T10:02:10Z', 'user=user456', 'e

...

['2026-03-05T10:00:12Z', 'endpoint=/product/432', 'ip=192.168.1.10', 'ms=180', 'status=200']
['2026-03-05T10:00:31Z', 'endpoint=/search', 'ip=192.168.1.12', 'ms=410', 'status=200']
['2026-03-05T10:01:02Z', 'endpoint=/search', 'ip=192.168.1.99', 'ms=3100', 'status=200']
['2026-03-05T10:02:25Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=950', 'status=500']
['2026-03-05T10:03:15Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=870', 'status=502']
['2026-03-05T10:04:45Z', 'endpoint=/product/432', 'ip=192.168.1.10', 'ms=2800', 'status=200']
['2026-03-05T10:06:00Z', 'endpoint=/checkout', 'ip=192.168.1.12', 'ms=2000', 'status=500']
['2026-03-05T10:07:10Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=125', 'status=401']
['2026-03-05T10:07:20Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=128', 'status=401']
['2026-03-05T10:08:10Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=450', 'status=200']
['2026-03-05T10:08:30Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=490', 'status=200']
['2026-03-05T10:10:02Z', 'e

Good API Records: 12
Bad API Records: 1

Bad App Records Example:


['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']
['2026-03-05T10:00:12Z', 'level=INFO', 'user=user123', 'action=VIEW_PRODUCT', 'ip=192.168.1.10', 'device=web', 'endpoint=/product/432', 'ms=180', 'status=200']
['2026-03-05T10:00:20Z', 'level=INFO', 'user=user222', 'action=LOGIN_SUCCESS', 'ip=192.168.1.12', 'device=android', 'endpoint=/login', 'ms=140', 'status=200']
['2026-03-05T10:00:31Z', 'level=INFO', 'user=user222', 'action=SEARCH', 'ip=192.168.1.12', 'device=android', 'endpoint=/search', 'ms=410', 'status=200']
['2026-03-05T10:01:02Z', 'level=WARN', 'user=user999', 'action=API_CALL', 'ip=192.168.1.99', 'device=web', 'endpoint=/search', 'ms=3100', 'status=200']
['2026-03-05T10:01:13Z', 'level=INFO', 'user=user123', 'action=ADD_TO_CART', 'ip=192.168.1.10', 'device=web', 'endpoint=/cart/add', 'ms=220', 'status=200']
['2026-03-05T10:01:40Z', 'level=INFO', 'user=user333', 'action=LO

{'ts': 'MALFORMED'}

Bad Auth Records Example:
{'ts': 'BROKEN_AUTH_LINE', 'user': 'user999', 'ip': '1.2.3.4'}


['2026-03-05T10:00:05Z', 'user=user123', 'event=LOGIN_SUCCESS', 'ip=192.168.1.10', 'reason=ok']
['2026-03-05T10:00:20Z', 'user=user222', 'event=LOGIN_SUCCESS', 'ip=192.168.1.12', 'reason=ok']
['2026-03-05T10:01:40Z', 'user=user333', 'event=LOGIN_SUCCESS', 'ip=192.168.1.45', 'reason=ok']
['2026-03-05T10:02:10Z', 'user=user456', 'event=LOGIN_SUCCESS', 'ip=192.168.1.12', 'reason=ok']
['2026-03-05T10:07:01Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:10Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:20Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:30Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:40Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:50Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=b

## Section 5: Basic Log Analysis

Now that our data has been cleaned and validated, we can start analyzing it.

In this section we will answer some basic analytical questions such as:

1. How many logs belong to each log level (INFO, ERROR, WARN)?
2. Which user actions occur most frequently?
3. Which users are most active in the system?

To perform these operations efficiently in Spark, we use **Pair RDDs**.

A Pair RDD is an RDD where each record is stored as a **key-value pair**.

Example:

("INFO", 1)
("ERROR", 1)
("INFO", 1)

Spark can aggregate these values using transformations such as **reduceByKey()**.

In [7]:
# ---------------------------------
# 1. Count log levels in app logs
# ---------------------------------

# Create Pair RDD (level,1)

level_pairs = good_app.map(lambda x: (x["level"], 1))

# Aggregate counts

level_counts = level_pairs.reduceByKey(lambda a, b: a + b)

print("Log Level Counts:")
for row in level_counts.collect():
    print(row)



# ---------------------------------
# 2. Count user actions
# ---------------------------------

action_pairs = good_app.map(lambda x: (x["action"], 1))

action_counts = action_pairs.reduceByKey(lambda a, b: a + b)

print("\nUser Action Counts:")
for row in action_counts.collect():
    print(row)



# ---------------------------------
# 3. Count activity per user
# ---------------------------------

user_pairs = good_app.map(lambda x: (x["user"], 1))

user_activity = user_pairs.reduceByKey(lambda a, b: a + b)

print("\nUser Activity Counts:")
for row in user_activity.collect():
    print(row)

Log Level Counts:


['2026-03-05T10:06:00Z', 'level=ERROR', 'user=user222', 'action=DATABASE_TIMEOUT', 'ip=192.168.1.12', 'device=android', 'endpoint=/checkout', 'ms=2000', 'status=500']
['2026-03-05T10:06:25Z', 'level=INFO', 'user=user123', 'action=LOGOUT_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/logout', 'ms=90', 'status=200']
['2026-03-05T10:07:01Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=130', 'status=401']
['2026-03-05T10:07:10Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=125', 'status=401']
['2026-03-05T10:07:20Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=128', 'status=401']
['2026-03-05T10:07:30Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=126', 'status=401']
['2026-03-05T10:07:40Z', 'level=ERROR', 'user=user888', '

('WARN', 4)
('INFO', 29)
('ERROR', 9)

User Action Counts:
('SEARCH', 9)
('ADD_TO_CART', 3)
('PAYMENT_FAILED', 3)
('CHECKOUT', 2)
('PAYMENT_SUCCESS', 1)
('LOGIN_FAILED', 5)
('LOGIN_SUCCESS', 6)
('VIEW_PRODUCT', 5)
('API_CALL', 4)
('DATABASE_TIMEOUT', 1)
('LOGOUT_SUCCESS', 3)

User Activity Counts:


['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']
['2026-03-05T10:00:12Z', 'level=INFO', 'user=user123', 'action=VIEW_PRODUCT', 'ip=192.168.1.10', 'device=web', 'endpoint=/product/432', 'ms=180', 'status=200']
['2026-03-05T10:00:20Z', 'level=INFO', 'user=user222', 'action=LOGIN_SUCCESS', 'ip=192.168.1.12', 'device=android', 'endpoint=/login', 'ms=140', 'status=200']
['2026-03-05T10:06:00Z', 'level=ERROR', 'user=user222', 'action=DATABASE_TIMEOUT', 'ip=192.168.1.12', 'device=android', 'endpoint=/checkout', 'ms=2000', 'status=500']['2026-03-05T10:00:31Z', 'level=INFO', 'user=user222', 'action=SEARCH', 'ip=192.168.1.12', 'device=android', 'endpoint=/search', 'ms=410', 'status=200']

['2026-03-05T10:01:02Z', 'level=WARN', 'user=user999', 'action=API_CALL', 'ip=192.168.1.99', 'device=web', 'endpoint=/search', 'ms=3100', 'status=200']
['2026-03-05T10:01:13Z', 'level=INFO', 'user=user123',

('user222', 7)
('user333', 6)
('user456', 4)
('user777', 3)
('user888', 6)
('user123', 6)
('user999', 2)
('user555', 1)
('user111', 7)


## Section 6: Fraud Detection

Large online platforms must continuously monitor for suspicious activities.

Some common fraud patterns include:

1. Repeated login failures (possible brute force attacks)
2. Repeated payment failures (possible fraud attempts)
3. Requests coming from blacklisted IP addresses

In this section, we will use Spark RDD transformations to detect such patterns.

We will:
- identify users with repeated login failures
- detect users with multiple failed payments
- find activity from blacklisted IPs

This is an example of real-world security analytics.

In [8]:
# ---------------------------------
# Login failures from auth logs
# ---------------------------------

login_failures = good_auth.filter(lambda x: x.get("event") == "LOGIN_FAILED")

# Create pair RDD (user,1)

login_failure_pairs = login_failures.map(lambda x: (x["user"], 1))

# Count failures per user

login_failure_counts = login_failure_pairs.reduceByKey(lambda a, b: a + b)

# Sort descending

login_failure_sorted = login_failure_counts.sortBy(lambda x: x[1], ascending=False)

print("Login Failure Counts:")
for row in login_failure_sorted.collect():
    print(row)

['2026-03-05T10:00:05Z', 'user=user123', 'event=LOGIN_SUCCESS', 'ip=192.168.1.10', 'reason=ok']
['2026-03-05T10:00:20Z', 'user=user222', 'event=LOGIN_SUCCESS', 'ip=192.168.1.12', 'reason=ok']
['2026-03-05T10:01:40Z', 'user=user333', 'event=LOGIN_SUCCESS', 'ip=192.168.1.45', 'reason=ok']
['2026-03-05T10:02:10Z', 'user=user456', 'event=LOGIN_SUCCESS', 'ip=192.168.1.12', 'reason=ok']
['2026-03-05T10:07:01Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:10Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:20Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:30Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:40Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=bad_password']
['2026-03-05T10:07:50Z', 'user=user888', 'event=LOGIN_FAILED', 'ip=192.168.1.55', 'reason=b

Login Failure Counts:
('user888', 6)
('user111', 2)


In [9]:
# ---------------------------------
# Payment failures
# ---------------------------------

failed_payments = good_payment.filter(lambda x: x.get("status") == "FAILED")

payment_failure_pairs = failed_payments.map(lambda x: (x["user"],1))

payment_failure_counts = payment_failure_pairs.reduceByKey(lambda a,b: a+b)

payment_failure_sorted = payment_failure_counts.sortBy(lambda x: x[1], ascending=False)

print("\nPayment Failure Counts:")
for row in payment_failure_sorted.collect():
    print(row)

['2026-03-05T10:10:30Z', 'txn=tx105', 'user=user222', 'amount=450', 'method=UPI', 'status=SUCCESS', 'ip=192.168.1.12']
['2026-03-05T10:10:40Z', 'txn=tx106', 'user=user111', 'amount=299', 'method=CARD', 'status=FAILED', 'ip=10.10.10.10']
['MALFORMED_PAYMENT', 'txn=tx999', 'user=user000', 'amount=', 'status=FAILED']
['2026-03-05T10:02:25Z', 'txn=tx101', 'user=user456', 'amount=1250', 'method=UPI', 'status=FAILED', 'ip=192.168.1.12']
['2026-03-05T10:03:15Z', 'txn=tx102', 'user=user456', 'amount=1250', 'method=UPI', 'status=FAILED', 'ip=192.168.1.12']
['2026-03-05T10:05:22Z', 'txn=tx103', 'user=user333', 'amount=899', 'method=CARD', 'status=SUCCESS', 'ip=192.168.1.45']
['2026-03-05T10:09:25Z', 'txn=tx104', 'user=user456', 'amount=1250', 'method=UPI', 'status=FAILED', 'ip=192.168.1.12']



Payment Failure Counts:
('user456', 3)
('user111', 1)


In [10]:
# ---------------------------------
# Blacklisted IP detection
# ---------------------------------

# Convert blacklist into set
blacklist = set(ip_blacklist.collect())

# Filter logs where IP is blacklisted

blacklisted_activity = good_app.filter(lambda x: x.get("ip") in blacklist)

print("\nBlacklisted IP Activity:")
for row in blacklisted_activity.take(5):
    print(row)


Blacklisted IP Activity:
{'ts': '2026-03-05T10:07:01Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '192.168.1.55', 'device': 'ios', 'endpoint': '/login', 'ms': '130', 'status': '401'}
{'ts': '2026-03-05T10:07:10Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '192.168.1.55', 'device': 'ios', 'endpoint': '/login', 'ms': '125', 'status': '401'}
{'ts': '2026-03-05T10:07:20Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '192.168.1.55', 'device': 'ios', 'endpoint': '/login', 'ms': '128', 'status': '401'}
{'ts': '2026-03-05T10:07:30Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '192.168.1.55', 'device': 'ios', 'endpoint': '/login', 'ms': '126', 'status': '401'}
{'ts': '2026-03-05T10:07:40Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '192.168.1.55', 'device': 'ios', 'endpoint': '/login', 'ms': '124', 'status': '401'}


['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']
['2026-03-05T10:00:12Z', 'level=INFO', 'user=user123', 'action=VIEW_PRODUCT', 'ip=192.168.1.10', 'device=web', 'endpoint=/product/432', 'ms=180', 'status=200']
['2026-03-05T10:00:20Z', 'level=INFO', 'user=user222', 'action=LOGIN_SUCCESS', 'ip=192.168.1.12', 'device=android', 'endpoint=/login', 'ms=140', 'status=200']
['2026-03-05T10:00:31Z', 'level=INFO', 'user=user222', 'action=SEARCH', 'ip=192.168.1.12', 'device=android', 'endpoint=/search', 'ms=410', 'status=200']
['2026-03-05T10:01:02Z', 'level=WARN', 'user=user999', 'action=API_CALL', 'ip=192.168.1.99', 'device=web', 'endpoint=/search', 'ms=3100', 'status=200']
['2026-03-05T10:01:13Z', 'level=INFO', 'user=user123', 'action=ADD_TO_CART', 'ip=192.168.1.10', 'device=web', 'endpoint=/cart/add', 'ms=220', 'status=200']
['2026-03-05T10:01:40Z', 'level=INFO', 'user=user333', 'action=LO

## Section 7: API Performance Monitoring

Modern applications rely heavily on APIs.

Every user action such as:
- login
- product search
- payment
- checkout

is handled by APIs.

Monitoring API performance is very important because slow or failing APIs directly impact user experience.

In this section we will analyze API gateway logs to answer the following questions:

1. Which APIs are called the most?
2. Which APIs are slow?
3. Which APIs are returning errors?
4. What is the average response time per API?

We will use Spark RDD transformations such as:

- map()
- filter()
- reduceByKey()
- sortBy()

to perform these analytics.

In [11]:
# ---------------------------------
# Count API calls per endpoint
# ---------------------------------

endpoint_pairs = good_api.map(lambda x: (x["endpoint"], 1))

endpoint_counts = endpoint_pairs.reduceByKey(lambda a, b: a + b)

endpoint_sorted = endpoint_counts.sortBy(lambda x: x[1], ascending=False)

print("API Call Counts:")
for row in endpoint_sorted.collect():
    print(row)

['2026-03-05T10:07:10Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=125', 'status=401']
['2026-03-05T10:00:12Z', 'endpoint=/product/432', 'ip=192.168.1.10', 'ms=180', 'status=200']['2026-03-05T10:07:20Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=128', 'status=401']

['2026-03-05T10:08:10Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=450', 'status=200']
['2026-03-05T10:08:30Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=490', 'status=200']
['2026-03-05T10:10:02Z', 'endpoint=/search', 'ip=192.168.1.99', 'ms=3400', 'status=200']['2026-03-05T10:00:31Z', 'endpoint=/search', 'ip=192.168.1.12', 'ms=410', 'status=200']

['GARBAGE_GATEWAY_LINE', 'something', 'weird', 'here']
['2026-03-05T10:01:02Z', 'endpoint=/search', 'ip=192.168.1.99', 'ms=3100', 'status=200']
['2026-03-05T10:02:25Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=950', 'status=500']
['2026-03-05T10:03:15Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=870', 'status=502']
['2026-03-05T10:04:45Z', 'endpoint=/product/432', 'ip=192.168.1.10',

API Call Counts:
('/search', 5)
('/product/432', 2)
('/pay', 2)
('/login', 2)
('/checkout', 1)


In [12]:
# ---------------------------------
# Detect slow APIs
# ---------------------------------

slow_api = good_api.filter(lambda x: int(x["ms"]) > 2000)

slow_api_pairs = slow_api.map(lambda x: (x["endpoint"], 1))

slow_api_counts = slow_api_pairs.reduceByKey(lambda a, b: a + b)

print("\nSlow API Counts:")
for row in slow_api_counts.collect():
    print(row)


Slow API Counts:
('/search', 2)
('/product/432', 1)


['2026-03-05T10:07:10Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=125', 'status=401']
['2026-03-05T10:07:20Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=128', 'status=401']
['2026-03-05T10:08:10Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=450', 'status=200']
['2026-03-05T10:08:30Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=490', 'status=200']
['2026-03-05T10:10:02Z', 'endpoint=/search', 'ip=192.168.1.99', 'ms=3400', 'status=200']
['GARBAGE_GATEWAY_LINE', 'something', 'weird', 'here']
['2026-03-05T10:00:12Z', 'endpoint=/product/432', 'ip=192.168.1.10', 'ms=180', 'status=200']
['2026-03-05T10:00:31Z', 'endpoint=/search', 'ip=192.168.1.12', 'ms=410', 'status=200']
['2026-03-05T10:01:02Z', 'endpoint=/search', 'ip=192.168.1.99', 'ms=3100', 'status=200']
['2026-03-05T10:02:25Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=950', 'status=500']
['2026-03-05T10:03:15Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=870', 'status=502']
['2026-03-05T10:04:45Z', 'endpoint=/product/432', 'ip=192.168.1.10',

In [13]:
# ---------------------------------
# Detect API errors
# ---------------------------------

error_api = good_api.filter(lambda x: int(x["status"]) >= 400)

error_api_pairs = error_api.map(lambda x: (x["endpoint"], 1))

error_api_counts = error_api_pairs.reduceByKey(lambda a, b: a + b)

print("\nAPI Error Counts:")
for row in error_api_counts.collect():
    print(row)


API Error Counts:


['2026-03-05T10:07:10Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=125', 'status=401']
['2026-03-05T10:07:20Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=128', 'status=401']
['2026-03-05T10:08:10Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=450', 'status=200']
['2026-03-05T10:08:30Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=490', 'status=200']
['2026-03-05T10:10:02Z', 'endpoint=/search', 'ip=192.168.1.99', 'ms=3400', 'status=200']
['GARBAGE_GATEWAY_LINE', 'something', 'weird', 'here']
['2026-03-05T10:00:12Z', 'endpoint=/product/432', 'ip=192.168.1.10', 'ms=180', 'status=200']
['2026-03-05T10:00:31Z', 'endpoint=/search', 'ip=192.168.1.12', 'ms=410', 'status=200']
['2026-03-05T10:01:02Z', 'endpoint=/search', 'ip=192.168.1.99', 'ms=3100', 'status=200']
['2026-03-05T10:02:25Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=950', 'status=500']
['2026-03-05T10:03:15Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=870', 'status=502']
['2026-03-05T10:04:45Z', 'endpoint=/product/432', 'ip=192.168.1.10',

('/pay', 2)
('/checkout', 1)
('/login', 2)


In [14]:
# ---------------------------------
# Average response time per endpoint
# ---------------------------------

endpoint_time_pairs = good_api.map(lambda x: (x["endpoint"], (int(x["ms"]), 1)))

endpoint_time_sum = endpoint_time_pairs.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

endpoint_avg_time = endpoint_time_sum.mapValues(
    lambda x: x[0] / x[1]
)

print("\nAverage API Response Time:")
for row in endpoint_avg_time.collect():
    print(row)


Average API Response Time:


['2026-03-05T10:00:12Z', 'endpoint=/product/432', 'ip=192.168.1.10', 'ms=180', 'status=200']
['2026-03-05T10:00:31Z', 'endpoint=/search', 'ip=192.168.1.12', 'ms=410', 'status=200']
['2026-03-05T10:01:02Z', 'endpoint=/search', 'ip=192.168.1.99', 'ms=3100', 'status=200']
['2026-03-05T10:02:25Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=950', 'status=500']
['2026-03-05T10:03:15Z', 'endpoint=/pay', 'ip=192.168.1.12', 'ms=870', 'status=502']
['2026-03-05T10:04:45Z', 'endpoint=/product/432', 'ip=192.168.1.10', 'ms=2800', 'status=200']
['2026-03-05T10:06:00Z', 'endpoint=/checkout', 'ip=192.168.1.12', 'ms=2000', 'status=500']
['2026-03-05T10:07:10Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=125', 'status=401']
['2026-03-05T10:07:20Z', 'endpoint=/login', 'ip=192.168.1.55', 'ms=128', 'status=401']
['2026-03-05T10:08:10Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=450', 'status=200']
['2026-03-05T10:08:30Z', 'endpoint=/search', 'ip=10.10.10.10', 'ms=490', 'status=200']
['2026-03-05T10:10:02Z', 'e

('/product/432', 1490.0)
('/search', 1570.0)
('/pay', 910.0)
('/checkout', 2000.0)
('/login', 126.5)


## Section 8: Data Enrichment Using Joins

So far we have analyzed logs individually.

However, real-world analytics often requires combining data from multiple datasets.

This process is called **data enrichment**.

Data enrichment means adding additional information to existing records.

For example:
- Adding user country information to log data
- Checking whether an IP address is blacklisted
- Combining transaction logs with user profiles

Spark allows us to combine datasets using **Pair RDD joins**.

In this section we will:

1. Convert datasets into Pair RDD format
2. Join user logs with user reference data
3. Join log data with blacklisted IP dataset
4. Identify suspicious users using enriched information

This is a common data engineering pattern used in real-world pipelines.

In [15]:
# ---------------------------------
# Prepare users reference dataset
# ---------------------------------

users_pairs = users_ref.map(lambda x: x.split(",")) \
                       .map(lambda x: (x[0], {"country": x[1], "risk": x[2]}))

print("User Reference Data:")
for row in users_pairs.collect():
    print(row)

User Reference Data:
('user123', {'country': 'IND', 'risk': 'normal'})
('user222', {'country': 'IND', 'risk': 'normal'})
('user333', {'country': 'IND', 'risk': 'normal'})
('user456', {'country': 'IND', 'risk': 'normal'})
('user555', {'country': 'IND', 'risk': 'normal'})
('user777', {'country': 'IND', 'risk': 'normal'})
('user888', {'country': 'USA', 'risk': 'high_risk'})
('user999', {'country': 'USA', 'risk': 'normal'})
('user111', {'country': 'RUS', 'risk': 'high_risk'})


In [16]:
# ---------------------------------
# Convert app logs into pair RDD
# ---------------------------------

app_user_pairs = good_app.map(lambda x: (x["user"], x))

print("App Logs Pair Example:")
for row in app_user_pairs.take(3):
    print(row)

App Logs Pair Example:
('user123', {'ts': '2026-03-05T10:00:05Z', 'level': 'INFO', 'user': 'user123', 'action': 'LOGIN_SUCCESS', 'ip': '192.168.1.10', 'device': 'web', 'endpoint': '/login', 'ms': '120', 'status': '200'})
('user123', {'ts': '2026-03-05T10:00:12Z', 'level': 'INFO', 'user': 'user123', 'action': 'VIEW_PRODUCT', 'ip': '192.168.1.10', 'device': 'web', 'endpoint': '/product/432', 'ms': '180', 'status': '200'})
('user222', {'ts': '2026-03-05T10:00:20Z', 'level': 'INFO', 'user': 'user222', 'action': 'LOGIN_SUCCESS', 'ip': '192.168.1.12', 'device': 'android', 'endpoint': '/login', 'ms': '140', 'status': '200'})


['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']
['2026-03-05T10:00:12Z', 'level=INFO', 'user=user123', 'action=VIEW_PRODUCT', 'ip=192.168.1.10', 'device=web', 'endpoint=/product/432', 'ms=180', 'status=200']
['2026-03-05T10:00:20Z', 'level=INFO', 'user=user222', 'action=LOGIN_SUCCESS', 'ip=192.168.1.12', 'device=android', 'endpoint=/login', 'ms=140', 'status=200']


In [17]:
# ---------------------------------
# Join logs with user reference
# ---------------------------------

user_enriched_logs = app_user_pairs.join(users_pairs)

print("Joined Records Example:")
for row in user_enriched_logs.take(3):
    print(row)

Joined Records Example:
('user777', ({'ts': '2026-03-05T10:04:05Z', 'level': 'INFO', 'user': 'user777', 'action': 'LOGIN_SUCCESS', 'ip': '192.168.1.77', 'device': 'web', 'endpoint': '/login', 'ms': '130', 'status': '200'}, {'country': 'IND', 'risk': 'normal'}))
('user777', ({'ts': '2026-03-05T10:04:20Z', 'level': 'INFO', 'user': 'user777', 'action': 'SEARCH', 'ip': '192.168.1.77', 'device': 'web', 'endpoint': '/search', 'ms': '520', 'status': '200'}, {'country': 'IND', 'risk': 'normal'}))
('user777', ({'ts': '2026-03-05T10:09:40Z', 'level': 'INFO', 'user': 'user777', 'action': 'VIEW_PRODUCT', 'ip': '192.168.1.77', 'device': 'web', 'endpoint': '/product/555', 'ms': '200', 'status': '200'}, {'country': 'IND', 'risk': 'normal'}))


['2026-03-05T10:06:00Z', 'level=ERROR', 'user=user222', 'action=DATABASE_TIMEOUT', 'ip=192.168.1.12', 'device=android', 'endpoint=/checkout', 'ms=2000', 'status=500']
['2026-03-05T10:06:25Z', 'level=INFO', 'user=user123', 'action=LOGOUT_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/logout', 'ms=90', 'status=200']
['2026-03-05T10:07:01Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=130', 'status=401']
['2026-03-05T10:07:10Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=125', 'status=401']
['2026-03-05T10:07:20Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=128', 'status=401']
['2026-03-05T10:07:30Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=126', 'status=401']
['2026-03-05T10:07:40Z', 'level=ERROR', 'user=user888', '

In [18]:
# ---------------------------------
# Detect high risk user activity
# ---------------------------------

high_risk_activity = user_enriched_logs.filter(
    lambda x: x[1][1]["risk"] == "high_risk"
)

print("High Risk User Activity:")
for row in high_risk_activity.take(5):
    print(row)

High Risk User Activity:
('user111', ({'ts': '2026-03-05T10:08:05Z', 'level': 'INFO', 'user': 'user111', 'action': 'LOGIN_SUCCESS', 'ip': '10.10.10.10', 'device': 'web', 'endpoint': '/login', 'ms': '140', 'status': '200'}, {'country': 'RUS', 'risk': 'high_risk'}))
('user111', ({'ts': '2026-03-05T10:08:10Z', 'level': 'INFO', 'user': 'user111', 'action': 'SEARCH', 'ip': '10.10.10.10', 'device': 'web', 'endpoint': '/search', 'ms': '450', 'status': '200'}, {'country': 'RUS', 'risk': 'high_risk'}))
('user111', ({'ts': '2026-03-05T10:08:15Z', 'level': 'INFO', 'user': 'user111', 'action': 'SEARCH', 'ip': '10.10.10.10', 'device': 'web', 'endpoint': '/search', 'ms': '460', 'status': '200'}, {'country': 'RUS', 'risk': 'high_risk'}))
('user111', ({'ts': '2026-03-05T10:08:20Z', 'level': 'INFO', 'user': 'user111', 'action': 'SEARCH', 'ip': '10.10.10.10', 'device': 'web', 'endpoint': '/search', 'ms': '470', 'status': '200'}, {'country': 'RUS', 'risk': 'high_risk'}))
('user111', ({'ts': '2026-03-05T1

In [19]:
# ---------------------------------
# Prepare blacklist pair RDD
# ---------------------------------

blacklist_pairs = ip_blacklist.map(lambda x: (x.strip(), "blacklisted"))

# Convert logs into (ip, log_record)

app_ip_pairs = good_app.map(lambda x: (x["ip"], x))

# Join

blacklisted_join = app_ip_pairs.join(blacklist_pairs)

print("Blacklisted IP Activity:")
for row in blacklisted_join.take(5):
    print(row)

Blacklisted IP Activity:


['2026-03-05T10:06:00Z', 'level=ERROR', 'user=user222', 'action=DATABASE_TIMEOUT', 'ip=192.168.1.12', 'device=android', 'endpoint=/checkout', 'ms=2000', 'status=500']
['2026-03-05T10:06:25Z', 'level=INFO', 'user=user123', 'action=LOGOUT_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/logout', 'ms=90', 'status=200']
['2026-03-05T10:07:01Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=130', 'status=401']
['2026-03-05T10:07:10Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=125', 'status=401']
['2026-03-05T10:07:20Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=128', 'status=401']
['2026-03-05T10:07:30Z', 'level=ERROR', 'user=user888', 'action=LOGIN_FAILED', 'ip=192.168.1.55', 'device=ios', 'endpoint=/login', 'ms=126', 'status=401']
['2026-03-05T10:07:40Z', 'level=ERROR', 'user=user888', '

('192.168.1.55', ({'ts': '2026-03-05T10:07:01Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '192.168.1.55', 'device': 'ios', 'endpoint': '/login', 'ms': '130', 'status': '401'}, 'blacklisted'))
('192.168.1.55', ({'ts': '2026-03-05T10:07:10Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '192.168.1.55', 'device': 'ios', 'endpoint': '/login', 'ms': '125', 'status': '401'}, 'blacklisted'))
('192.168.1.55', ({'ts': '2026-03-05T10:07:20Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '192.168.1.55', 'device': 'ios', 'endpoint': '/login', 'ms': '128', 'status': '401'}, 'blacklisted'))
('192.168.1.55', ({'ts': '2026-03-05T10:07:30Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '192.168.1.55', 'device': 'ios', 'endpoint': '/login', 'ms': '126', 'status': '401'}, 'blacklisted'))
('192.168.1.55', ({'ts': '2026-03-05T10:07:40Z', 'level': 'ERROR', 'user': 'user888', 'action': 'LOGIN_FAILED', 'ip': '1

## Section 9: Final Reports and Analytics Output

In previous sections we performed multiple analyses on system logs.

We:
- parsed raw logs
- cleaned malformed records
- counted log events
- detected suspicious login failures
- detected payment failures
- monitored API performance
- enriched logs with user and IP reference data

In this section we combine these insights to generate final reports.

The final outputs include:

1. Fraud Alert Report
2. API Performance Report
3. Suspicious User Summary

These reports simulate the outputs produced by real log analytics systems.

In [20]:
# ---------------------------------
# Fraud Alert Report
# ---------------------------------

# Login failure alerts
login_alerts = login_failure_counts.map(
    lambda x: ("LOGIN_FAILURE_ALERT", x[0], x[1])
)

# Payment failure alerts
payment_alerts = payment_failure_counts.map(
    lambda x: ("PAYMENT_FAILURE_ALERT", x[0], x[1])
)

# Combine alerts
fraud_alerts = login_alerts.union(payment_alerts)

print("Fraud Alert Report:")
for row in fraud_alerts.collect():
    print(row)

Fraud Alert Report:
('LOGIN_FAILURE_ALERT', 'user888', 6)
('LOGIN_FAILURE_ALERT', 'user111', 2)
('PAYMENT_FAILURE_ALERT', 'user456', 3)
('PAYMENT_FAILURE_ALERT', 'user111', 1)


In [21]:
# ---------------------------------
# API Performance Summary
# ---------------------------------

api_performance = endpoint_avg_time.join(endpoint_counts)

print("\nAPI Performance Report:")
for row in api_performance.collect():
    print(row)


API Performance Report:
('/product/432', (1490.0, 2))
('/search', (1570.0, 5))
('/pay', (910.0, 2))
('/checkout', (2000.0, 1))
('/login', (126.5, 2))


In [22]:
# ---------------------------------
# Suspicious User Summary
# ---------------------------------

suspicious_users = login_failure_counts.union(payment_failure_counts)

suspicious_summary = suspicious_users.reduceByKey(lambda a,b: a+b)

print("\nSuspicious User Summary:")
for row in suspicious_summary.collect():
    print(row)


Suspicious User Summary:
('user888', 6)
('user456', 3)
('user111', 3)


In [23]:
top_suspicious_users = suspicious_summary.sortBy(lambda x: x[1], ascending=False)

print("\nTop Suspicious Users:")
for row in top_suspicious_users.collect():
    print(row)


Top Suspicious Users:
('user888', 6)
('user456', 3)
('user111', 3)


In [24]:
fraud_alerts.saveAsTextFile("output/fraud_alerts")

endpoint_counts.saveAsTextFile("output/api_call_counts")

endpoint_avg_time.saveAsTextFile("output/api_avg_response_time")

top_suspicious_users.saveAsTextFile("output/suspicious_users")